# 00 — Colab setup and data verification
Run this once per session before the others.


## Colab setup
Verify GPU, mount Drive, clone the repo, install requirements, and create the project tree on Drive.


In [ ]:
# 1. GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Mount Drive (skipped automatically when not on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/cs4782_patchtst_project'
    IN_COLAB = True
except Exception:
    PROJECT_ROOT = '.'
    IN_COLAB = False
print('PROJECT_ROOT =', PROJECT_ROOT, ' IN_COLAB =', IN_COLAB)


In [ ]:
# 3. Clone repo (Colab only). Update REPO_URL in scripts/build_notebooks.py
# and re-run that script to regenerate notebooks if the URL changes.
REPO_URL = 'https://github.com/Ash1R/ATSW64W-experiments.git'
if IN_COLAB:
    import os, subprocess
    os.chdir('/content')
    # Derive the clone directory from the URL's basename so it matches the repo name.
    REPO_DIRNAME = REPO_URL.rstrip('/').rsplit('/', 1)[-1]
    if REPO_DIRNAME.endswith('.git'):
        REPO_DIRNAME = REPO_DIRNAME[:-4]
    if not os.path.isdir(f'/content/{REPO_DIRNAME}'):
        # check=True so a bad URL fails loudly here instead of crashing the next chdir.
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIRNAME], check=True)
    os.chdir(f'/content/{REPO_DIRNAME}')
    subprocess.run(['git', 'pull'], check=False)
print('cwd =', __import__('os').getcwd())


In [ ]:
# 4. Install requirements (best-effort; resolved relative to the repo root
# regardless of where the kernel started, so headless `nbconvert` runs work).
import subprocess, sys, os
_req_dir = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_req_dir, 'requirements.txt')):
        break
    _req_dir = os.path.dirname(_req_dir)
_req = os.path.join(_req_dir, 'requirements.txt')
if os.path.isfile(_req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req], check=False)
else:
    print('skipping pip install — requirements.txt not found from', os.getcwd())


In [ ]:
# 5. Make the project's `code/` directory importable.
# We add `code/` itself to sys.path (not the repo root) because the
# stdlib already ships a module named `code` that the IPython kernel
# imports before this cell runs — shadowing that cleanly is messy.
# This way every import is `from utils...`, `from data...`, `from models...`.
import sys, os
# When run with `jupyter nbconvert --execute`, the kernel's cwd is
# the notebook's directory (`notebooks/`), so locate the repo root
# by walking up until we find `code/`. On Colab we already chdir'd
# into the cloned repo above.
REPO_DIR = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(REPO_DIR, 'code')):
        break
    REPO_DIR = os.path.dirname(REPO_DIR)
CODE_DIR = os.path.join(REPO_DIR, 'code')
if not os.path.isdir(CODE_DIR):
    raise RuntimeError(f'could not locate code/ from {os.getcwd()}')
os.chdir(REPO_DIR)
for p in (REPO_DIR, CODE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
print('REPO_DIR =', REPO_DIR)
from utils.colab import ensure_dirs
subdirs = ensure_dirs(PROJECT_ROOT)
for k, v in subdirs.items():
    print(f'{k:>12}  {v}')


## Verify a dataset CSV exists and load one window


In [ ]:
from data.dataset import build_data_bundle
bundle = build_data_bundle(PROJECT_ROOT, 'weather', seq_len=336, pred_len=96)
print('weather: num_channels =', bundle.num_channels,
      ' train =', len(bundle.train), ' val =', len(bundle.val), ' test =', len(bundle.test))


In [ ]:
from torch.utils.data import DataLoader
loader = DataLoader(bundle.train, batch_size=8, shuffle=False)
x, y = next(iter(loader))
print('x shape:', tuple(x.shape))   # [B, 336, M]
print('y shape:', tuple(y.shape))   # [B, 96, M]


If the cell above raised `FileNotFoundError`, follow `data/README.md` and place the CSV under `<PROJECT_ROOT>/data/weather/weather.csv` on Drive, then re-run.


## (Optional) Verify Electricity


In [ ]:
try:
    eb = build_data_bundle(PROJECT_ROOT, 'electricity', seq_len=336, pred_len=96)
    print('electricity: num_channels =', eb.num_channels,
          ' train =', len(eb.train), ' val =', len(eb.val), ' test =', len(eb.test))
except FileNotFoundError as e:
    print('Electricity not present yet:', e)
